# Notebook to run a comparison of models exported to ONNX files

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils
import Evaluation

from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix, classification_report, accuracy_score, average_precision_score, log_loss, auc
from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax
from scipy.stats import ks_2samp
from sklearn.inspection import permutation_importance
import seaborn as sns
from sklearn.calibration import CalibrationDisplay
from matplotlib import cm


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

import onnx
import numpy as np
from onnx import helper, numpy_helper, TensorProto
import onnxruntime as ort
import Utils
import matplotlib.pyplot as plt
import importlib
import pandas as pd
import seaborn as sns
import Evaluation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
importlib.reload(Utils);
importlib.reload(Evaluation)

# Read in & initialize

In [ ]:
df_OO = Utils.get_dataframe("Data/OOwmatchattempts.root", folder_name="DF_*")
df_PbPb = Utils.get_dataframe("Data/PbPbwmatchattempts.root", folder_name="DF_*")


In [ ]:
df_OO = Utils.process_dataframe(df_OO, makedummies=False)
df_PbPb = Utils.process_dataframe(df_PbPb, makedummies=False)

In [ ]:
# df_OO = Utils.subsample(df_OO, frac = 0.6)
# df_PbPb = Utils.subsample(df_PbPb, frac = 0.6)
# Subsample due to memory conditions, about a third for now

_, _, df_OO = Evaluation.Splitter(df_OO, val_frac=0.1, test_frac = 0.3) 
_, _, df_PbPb = Evaluation.Splitter(df_PbPb, val_frac=0.1, test_frac = 0.3) 
# Grab only the test set for unbiased evaluation





In [ ]:
# NOTE: current RABS cut added for uuuh rigour
df_OO = df_OO[(df_OO['Rabs']> 30) & (df_OO['Rabs']<80)]
df_PbPb = df_PbPb[(df_PbPb['Rabs']> 30) & (df_PbPb['Rabs']<80)]

In [ ]:
# FEATURES_OO = ['DeltaDirection', 'PullPt', 'APullPhi', 'PullTanl', 'PtMFT',
#        'CPhiPhiMFT', 'DeltaR', 'SameSign', 'DeltaEta', 'DeltaTanl',
#        'RelPtDiff', 'CYYMFT', 'C1Pt1PtMFT', 'PullPhi', 'CXXMFT',
#        'C1PtPhiMFT', 'YMCH', 'XMCH', 'DeltaPt', 'ADeltaPhi', 'PullR',
#        'PullX', 'PullY', 'CXYMFT', 'CTglTglMCH', 'DeltaPhi', 'C1PtXMFT']

# FEATURES_PBPB = ['RelPtDiff', 'SameSign', 'PtMFT', 'PullPt', 'CPhiPhiMFT',
#        'C1Pt1PtMFT', 'CTglTglMCH', 'CPhiPhiMCH', 'TanlMFT', 'CXXMFT',
#        'CYYMFT', 'DeltaDirection', 'DeltaR', 'ADeltaPhi', 'etaMFT',
#        'PullR', 'DeltaPt', 'C1PtPhiMFT', 'DeltaEta', 'ADeltaX',
#        'APullPhi', 'DeltaTanl', 'CYYMCH', 'ADeltaY', 'CTglTglMFT',
#        'PullTanl', 'CXXMCH', 'InvQPtMFT', 'PtMCH', 'C1Pt1PtMCH', 'PullY',
#        'CXYMFT', 'APullX', 'DeltaX', 'APullY', 'DeltaPhi', 'C1PtXMFT',
#        'DeltaY', 'CTglXMCH', 'etaMCH', 'PullX', 'YMCH', 'TanlMCH', 'XMCH',
#        'CPhiXMFT', 'CTglXMFT', 'CPhiYMFT', 'C1PtYMFT', 'CTglPhiMFT',
#        'PullPhi', 'XMFT', 'YMFT', 'CPhiYMCH', 'CTglYMFT', 'PhiMFT',
#        'PhiMCH', 'C1PtTglMCH', 'CTglYMCH', 'C1PtTglMFT', 'CPhiXMCH']

FEATURES = ['XMCH',
 'YMCH',
 'PhiMCH',
 'TanlMCH',
 'InvQPtMCH',
 'CXXMCH',
 'CYYMCH',
 'CPhiPhiMCH',
 'CTglTglMCH',
 'C1Pt1PtMCH',
 'CXYMCH',
 'CPhiYMCH',
 'CPhiXMCH',
 'CTglXMCH',
 'CTglYMCH',
 'CTglPhiMCH',
 'C1PtXMCH',
 'C1PtYMCH',
 'C1PtPhiMCH',
 'C1PtTglMCH',
 'XMFT',
 'YMFT',
 'PhiMFT',
 'TanlMFT',
 'InvQPtMFT',
 'TrackTypeMFT',
 'CXXMFT',
 'CYYMFT',
 'CPhiPhiMFT',
 'CTglTglMFT',
 'C1Pt1PtMFT',
 'CXYMFT',
 'CPhiYMFT',
 'CPhiXMFT',
 'CTglXMFT',
 'CTglYMFT',
 'CTglPhiMFT',
 'C1PtXMFT',
 'C1PtYMFT',
 'C1PtPhiMFT',
 'C1PtTglMFT',
 'etaMCH',
 'etaMFT',
 'DeltaEta',
 'DeltaX',
 'DeltaY',
 'DeltaPhi',
 'ADeltaPhi',
 'ADeltaX',
 'ADeltaY',
 'DeltaTanl',
 'DeltaR',
 'RMFT',
 'SameSign',
 'PtMCH',
 'PtMFT',
 'DeltaPt',
 'RelPtDiff',
 'PullPt',
 'PullX',
 'PullY',
 'PullR',
 'PullPhi',
 'PullTanl',
 'APullX',
 'APullY',
 'APullPhi',
 'DeltaDirection']

GROUP = "mchID" 

MODEL_OO = "lgbmOOallfeaturespt03.onnx"

MODEL_PBPB = "lgbmpbpballfeaturespt03.onnx"

model_pbpb_xgboost = "model_PbPb_FULL_XGB.onnx"

# Model evaluations

In [ ]:
df_OO = Evaluation.onnxinferlgbm(df_OO, FEATURES, MODEL_OO, 'score')
df_PbPb = Evaluation.onnxinferlgbm(df_PbPb, FEATURES, MODEL_PBPB, 'score')

In [ ]:
df_OO=Evaluation.onnxinferlgbm(df_OO, FEATURES, MODEL_PBPB, 'score_pbpb')
df_PbPb=Evaluation.onnxinferlgbm(df_PbPb, FEATURES, MODEL_OO, 'score_oo')

In [ ]:
sess = ort.InferenceSession(model_pbpb_xgboost)
features_xgb = ['XMCH', 'YMCH', 'PhiMCH', 'TanlMCH', 'InvQPtMCH', 'CXXMCH', 'CYYMCH', 'CPhiPhiMCH', 'CTglTglMCH', 'C1Pt1PtMCH', 'CXYMCH', 'CPhiYMCH', 'CPhiXMCH', 'CTglXMCH', 'CTglYMCH', 'CTglPhiMCH', 'C1PtXMCH', 'C1PtYMCH', 'C1PtPhiMCH', 'C1PtTglMCH', 'XMFT', 'YMFT', 'PhiMFT', 'TanlMFT', 'InvQPtMFT', 'TrackTypeMFT', 'CXXMFT', 'CYYMFT', 'CPhiPhiMFT', 'CTglTglMFT', 'C1Pt1PtMFT', 'CXYMFT', 'CPhiYMFT', 'CPhiXMFT', 'CTglXMFT', 'CTglYMFT', 'CTglPhiMFT', 'C1PtXMFT', 'C1PtYMFT', 'C1PtPhiMFT', 'C1PtTglMFT', 'etaMCH', 'etaMFT', 'DeltaEta', 'DeltaX', 'DeltaY', 'DeltaPhi', 'ADeltaPhi', 'ADeltaX', 'ADeltaY', 'DeltaTanl', 'DeltaR', 'RMFT', 'SameSign', 'PtMCH', 'PtMFT', 'DeltaPt', 'RelPtDiff', 'PMCH', 'PMFT', 'DeltaP', 'RelPDiff', 'PullX', 'PullY', 'PullR', 'PullPhi', 'PullTanl', 'APullX', 'APullY', 'APullPhi', 'PullPt', 'DeltaDirection']
input_name = sess.get_inputs()[0].name
df_PbPb['score_xgb'] = sess.run(
    None,
    {input_name: df_PbPb[features_xgb].to_numpy(dtype=np.float32)}
)[1][:,1]


In [ ]:
len(features_xgb)

# Lengthy NN addition for a single bloody plot

In [ ]:
FEATURES_NN = ['XMCH',
 'YMCH',
 'PhiMCH',
 'TanlMCH',
 'InvQPtMCH',
 'CXXMCH',
 'CYYMCH',
 'CPhiPhiMCH',
 'CTglTglMCH',
 'C1Pt1PtMCH',
 'CXYMCH',
 'CPhiYMCH',
 'CPhiXMCH',
 'CTglXMCH',
 'CTglYMCH',
 'CTglPhiMCH',
 'C1PtXMCH',
 'C1PtYMCH',
 'C1PtPhiMCH',
 'C1PtTglMCH',
 'XMFT',
 'YMFT',
 'PhiMFT',
 'TanlMFT',
 'InvQPtMFT',
 'TrackTypeMFT',
 'CXXMFT',
 'CYYMFT',
 'CPhiPhiMFT',
 'CTglTglMFT',
 'C1Pt1PtMFT',
 'CXYMFT',
 'CPhiYMFT',
 'CPhiXMFT',
 'CTglXMFT',
 'CTglYMFT',
 'CTglPhiMFT',
 'C1PtXMFT',
 'C1PtYMFT',
 'C1PtPhiMFT',
 'C1PtTglMFT',
 'etaMCH',
 'etaMFT',
 'DeltaEta',
 'is_dummy',
 'DeltaX',
 'DeltaY',
 'DeltaPhi',
 'ADeltaPhi',
 'ADeltaX',
 'ADeltaY',
 'DeltaTanl',
 'DeltaR',
 'RMFT',
 'SameSign',
 'PtMCH',
 'PtMFT',
 'DeltaPt',
 'RelPtDiff',
 'PullPt',
 'PullX',
 'PullY',
 'PullR',
 'PullPhi',
 'PullTanl',
 'APullX',
 'APullY',
 'APullPhi',
 'DeltaDirection']

In [ ]:
df_PbPb['is_dummy'] = 0
len(FEATURES_NN)

In [ ]:
# NN additions
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import joblib

# ── Hyperparameters ───────────────────────────────────────────────────────────
HIDDEN_LAYERS = [32, 32]
DROPOUT       = 0.2
BATCH_SIZE    = 4096
N_EPOCHS      = 50
LR            = 1e-3
PATIENCE      = 3    # early stopping

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# ── Model definition ──────────────────────────────────────────────────────────
class TrackMatchMLP(nn.Module):
    def __init__(self, n_features: int, hidden_layers: list, dropout: float):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden_layers:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))   # single logit output
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)          # (batch,)

In [ ]:
scaler = joblib.load("track_matching_scaler.pkl")
model = TrackMatchMLP(len(FEATURES_NN), [32, 32], 0.2)
model.load_state_dict(torch.load("track_matching_nn.pt", map_location="cpu"))
model.eval()

X = scaler.transform(df_PbPb[FEATURES_NN])
with torch.no_grad():
    df_PbPb["score_nn"] = torch.sigmoid(model(torch.tensor(X, dtype=torch.float32))).numpy()

In [ ]:
df_PbPb['score_nn'].describe()

# 2D Evaluation

## PBPB

In [ ]:
Utils.plot_metrics_vs_xy(df_PbPb,
    feature_x="PMCH",
    # fmin_x = 0.3,
    fmax_x = 35,
    feature_y="MatchAttempts",
    fmin_y = 0.0,
    fmax_y = 5000.0,
    threshold=0.1,
    metrics_fn=Utils.inhousemetrics,
    x_bins=5,
    y_bins=5,
    metric_col_prefix = 'score',
    min_entries=1500)

## OO

In [ ]:
Utils.plot_metrics_vs_xy(df_OO,
    feature_x="PMCH",
    # fmin_x = 0.3,
    fmax_x = 80,
    feature_y="MatchAttempts",
    # fmin_y = 0.0,
    fmax_y = 300.0,
    threshold=0.9,
    metrics_fn=Utils.inhousemetrics,
    x_bins=4,
    y_bins=4,
    min_entries=250,)

#TODO: Develop a JSOn based output&input for the model evaluation on O2Physics

## Models Diff

### In OO Data

In [ ]:
Utils.plot_metrics_vs_xy_diff(
    # NOTE: MINIMUMS NOT OPTIONAL HERE, OTHERWISE WE END UP WITH NON-OVERLAPPING BINS
    # NOTE 2: Purities have non-common dens due to its definition - gm rec true / gm rec
    feature_x="PMCH",
    feature_y="MatchAttempts",
    x_bins=5,
    y_bins=5,
    fmin_x=4,
    fmax_x=50,
    fmin_y=0,
    fmax_y=200,
    Nsigma=1.0,
    metrics_fn=Utils.inhousemetrics,
    df_a=df_OO, threshold_a=0.6, metric_col_prefix_a="score",
    df_b=df_OO, threshold_b=0.5, metric_col_prefix_b="score_pbpb",
    cmap="RdBu", relative=False, min_entries=1500, vmin=-0.1, vmax = 0.1,
)

PbPb model is "strictly worse" in the sense that it performs worse in every measure, BUT the amount is ~2%, possibly not statistically significant

### In PbPb data

In [ ]:
Utils.plot_metrics_vs_xy_diff(
    feature_x="PMCH",
    feature_y="MatchAttempts",
    x_bins=5,
    y_bins=5,
    fmin_x=4,
    fmax_x=20,
    fmin_y=0,
    fmax_y=5000,
    Nsigma=1.0,
    metrics_fn=Utils.inhousemetrics,
    df_a=df_PbPb, threshold_a=0.82, metric_col_prefix_a="score",
    df_b=df_PbPb, threshold_b=0.3, metric_col_prefix_b="score_oo",
    cmap="RdBu", relative=False, min_entries=5000, vmin=-0.15,vmax=0.15,
)

#1D

Purity & True eff are almost strictly worse - mainly at low pt & high mult

## Similar region comparison #02

In [ ]:
Utils.plot_metrics_vs_xy_diff(
    feature_x="PMCH",
    feature_y="MatchAttempts",
    x_bins=3,
    y_bins=3,
    fmin_x=4,
    fmax_x=20,
    fmin_y=0,
    fmax_y=200,
    Nsigma=0.0,
    metrics_fn=Utils.inhousemetrics,
    df_a=df_OO, threshold_a=0.6, metric_col_prefix_a="score",
    df_b=df_PbPb, threshold_b=0.45, metric_col_prefix_b="score",
    cmap="RdBu", relative=False, min_entries=500, vmin=-0.08,vmax=0.08
)

In [ ]:
Utils.plot_metrics_vs_xy(df_PbPb,
    feature_x="PMCH",
    fmin_x = 0,
    fmax_x = 35,
    feature_y="PhiMCH",
    fmin_y = -np.pi,
    fmax_y = np.pi,
    threshold=0.1,
    metrics_fn=Utils.inhousemetrics,
    x_bins=4,
    y_bins=20,
    metric_col_prefix = 'score',
    min_entries=1000)

# Metric score sweeps

## In OO data

In [ ]:
df_OO["score_softmax"] = (
    df_OO.groupby(GROUP)["score"]
    .transform(lambda s: softmax(s))
)
df_PbPb["score_softmax"] = (
    df_PbPb.groupby(GROUP)["score"]
    .transform(lambda s: softmax(s))
)

In [ ]:
importlib.reload(Utils)
Utils.sweep_threshold_plot(df_eval= df_OO, metrics_fn = Utils.inhousemetrics, title= "OO Data OO model" +" vs Score Threshold", score_col='score', Nsigma=3.0, n_steps=50)
Utils.sweep_threshold_plot(df_eval= df_OO, metrics_fn = Utils.inhousemetrics, title= "OO Data PBPB model" +" vs Score Threshold", score_col='score_pbpb', Nsigma=3.0, n_steps=50)


# TODO: add breakdowns in mult regions & pt

### Low Mult ____ we only have low mult

## In PbPb data

In [ ]:
Utils.sweep_threshold_plot(df_eval= df_PbPb, metrics_fn = Utils.inhousemetrics, title= "PbPb Data PBPB model" +" vs Score Threshold", score_col='score', Nsigma=3.0, n_steps=25)
Utils.sweep_threshold_plot(df_eval= df_PbPb, metrics_fn = Utils.inhousemetrics, title= "PbPb Data OO model" +" vs Score Threshold", score_col='score_oo', Nsigma=3.0, n_steps=50)
Utils.sweep_threshold_plot(df_eval= df_PbPb, metrics_fn = Utils.inhousemetrics, title= "PbPb Data PbPb XGBoost model" +" vs Score Threshold", score_col='score_xgb', Nsigma=3.0, n_steps=25)

#TODO add breakdown plots for similar regions, e.g. similar multiplicity regions

### Low mult

In [ ]:
Utils.sweep_threshold_plot(df_eval= df_PbPb[df_PbPb['MatchAttempts']<300], metrics_fn = Utils.inhousemetrics, title= " Low Mult PbPb Data PBPB model" +" vs Score Threshold", score_col='score', Nsigma=3.0, n_steps=50)
Utils.sweep_threshold_plot(df_eval= df_PbPb[df_PbPb['MatchAttempts']<300], metrics_fn = Utils.inhousemetrics, title= "Low Mult PbPb Data OO model" +" vs Score Threshold", score_col='score_oo', Nsigma=3.0, n_steps=50)

### Intermediate Mult

In [ ]:
Utils.sweep_threshold_plot(df_eval= df_PbPb[(df_PbPb['MatchAttempts'] > 1000) & (df_PbPb['MatchAttempts']<2000)], metrics_fn = Utils.inhousemetrics, title= "mid Mult PbPb Data PBPB model" +" vs Score Threshold", score_col='score', Nsigma=3.0, n_steps=50)
Utils.sweep_threshold_plot(df_eval= df_PbPb[(df_PbPb['MatchAttempts'] > 1000) & (df_PbPb['MatchAttempts']<2000)], metrics_fn = Utils.inhousemetrics, title= "mid Mult PbPb Data OO model" +" vs Score Threshold", score_col='score_oo', Nsigma=3.0, n_steps=50)

### High Multiplicity

In [ ]:
Utils.sweep_threshold_plot(df_eval= df_PbPb[df_PbPb['MatchAttempts']>3000], metrics_fn = Utils.inhousemetrics, title= "High Mult PbPb Data PBPB model" +" vs Score Threshold", score_col='score', Nsigma=3.0, n_steps=50)
Utils.sweep_threshold_plot(df_eval= df_PbPb[df_PbPb['MatchAttempts']>3000], metrics_fn = Utils.inhousemetrics, title= "High Mult PbPb Data OO model" +" vs Score Threshold", score_col='score_oo', Nsigma=3.0, n_steps=50)

# Score distributions

## OO


In [ ]:
df_OO[['MatchAttempts','PtMCH']].describe()


In [ ]:
ooptmed = df_OO['PtMCH'].median()
oomamed = df_OO['MatchAttempts'].median()

### Mult

In [ ]:
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO), feature="score", title="combined score distribution", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO[(df_OO['MatchAttempts']>oomamed) ]), feature="score", title="high mult", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO[(df_OO['MatchAttempts']<oomamed) ]), feature="score", title="low mult", log = True, density = False)
# for entry in Utils.GROUP_PRESERVING_FEATURES:   
#     Utils.plot_metrics_vs_feature(df=df_OO,feature=entry, threshold = 0.8, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.0, trim_high=0., Nsigma=1.0)

# & (df_OO['PtMCH']>ooptmed)

Notice the change in where the distributions cut one another; True&Wrong from 0.8->0.5; wrong+fake & True 0.3-0.5->0.1-0.2

Also note the relatvie number of each group within each plot changes rapidly

### pt

In [ ]:
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO), feature="score", title="df OO model OO", log = True, density = True)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb), feature="score_oo", title="df PbPb model OO", log = True, density = True)

Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO[(df_OO['PtMCH']>ooptmed) ]), feature="score", title="high pt", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_OO[(df_OO['PtMCH']<ooptmed)]), feature="score", title="low pt", log = True, density = False)

distributions continue to remain largely similar, but groups changed prevalence 

## PbPb

In [ ]:
df_PbPb[['MatchAttempts','PtMCH']].describe()

In [ ]:
pbptmed = df_PbPb['PtMCH'].median()
pbmamed = df_PbPb['MatchAttempts'].median() #.quantile(0.25)

### Mult

In [ ]:
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb), feature="score", title="combined score distribution", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb), feature="score_xgb", title="combined score distribution - XGB", log = True, density = False)

Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb[(df_PbPb['MatchAttempts']>pbmamed) ]), feature="score", title="high mult", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb[(df_PbPb['MatchAttempts']<pbmamed) ]), feature="score", title="low mult", log = True, density = False)

Virtually no change based on this multiplicity split

### pt

In [ ]:
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb), feature="score", title="combined score distribution", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb[(df_PbPb['PtMCH']>pbptmed) ]), feature="score", title="high pt", log = True, density = False)
Utils.draw_feature(match_groups=Utils.build_match_groups(df_PbPb[(df_PbPb['PtMCH']<pbptmed)]), feature="score", title="low pt", log = True, density = False)

At low pt we can note a creep of the Wrong matches up to high scores

At low pt we see that the score distribution of True matches is missing its strong peak

# Leading score breakdowns

## OO

In [ ]:
Evaluation.plotleadingmatch(df_OO, metrics=['score', 'score_pbpb'], log = True, density = False)

## PbPb

In [ ]:
Evaluation.plotleadingmatch(df_PbPb, metrics=['score', 'score_oo'], log = True, density = False)

# Binned True Match score plots

## OO

In [ ]:
df_OO['logmatchattempts'] = np.log10(df_OO['MatchAttempts'])
df_PbPb['logmatchattempts'] = np.log10(df_PbPb['MatchAttempts'])

In [ ]:
feature2breakdown = 'PMCH'# 'PMCH'#'MatchAttempts'#, 'PMCH', 'MatchAttempts'

In [ ]:
Evaluation.featuredecompositionplot(df_OO[df_OO['IsSignal']==1], featureplot='score', featurebreakdown=feature2breakdown,equalwidth=True, n_bins=3, density=False)
Evaluation.featuredecompositionplot(df_OO[df_OO['IsSignal']==1], featureplot='score_pbpb', featurebreakdown=feature2breakdown,equalwidth=True, n_bins=3, density=False)


## PbPb

In [ ]:
Evaluation.featuredecompositionplot(df_PbPb[df_PbPb['IsSignal']==1], featureplot='score', featurebreakdown=feature2breakdown,equalwidth=True, n_bins=3, density=False)
Evaluation.featuredecompositionplot(df_PbPb[df_PbPb['IsSignal']==1], featureplot='score_oo', featurebreakdown=feature2breakdown,equalwidth=True, n_bins=3, density=False)

There is a very clear dependence of the score distributions on ... several features, such as:
* P/PtMCH
* Rabs - milder
* Not an awful lot of dependence on MatchAttempts

# Featurewise metric sweeps

In [ ]:
group_preserving_features = [ 'PhiMCH','MatchAttempts', 'Rabs','PMCH'] # 'etaMCH','MatchAttempts', 'Rabs', 'PtMCH', 'PMCH', 


## OO

Overall solid performance, TP~0.97, purity 0.99 at the optimal highptlowmatchattempts bin

In [ ]:
for entry in group_preserving_features:   
    Utils.plot_metrics_vs_feature(df=df_OO[df_OO['MatchAttempts']<300],feature=entry, threshold = 0.5, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=50, trim_low=0.0, trim_high=0.0, Nsigma=1.0) 
# NOTE: RABS adds + logmomemtum probably 

## PbPb
At low MatchAttempts in this high pt bin we can retrieve respectable, but not as good as OO, performance with both key metrics > 0.9

NOTE: Still with the missing matches included, so a ~5% degradation is not unexpected

At higher matchattempts we do see the usual degradation, but not as bad as in the combined case

In [ ]:
df_PbPb['RMCH'] = np.sqrt(df_PbPb['XMCH']**2+df_PbPb['YMCH']**2)
[]

In [ ]:
# df_PbPb_highpt = df_PbPb[(df_PbPb['PtMCH'] > 1.0) ]
importlib.reload(Utils);
for entry in ['RMCH']+Utils.GROUP_PRESERVING_FEATURES :
    Utils.plot_metrics_vs_feature(df=df_PbPb,feature=entry, threshold = 0.5, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=50, trim_low=0.01, trim_high=0.01, Nsigma=1.0) 
    # Utils.plot_metrics_vs_feature(df=df_PbPb[df_PbPb['MatchAttempts']<300],feature=entry, threshold = 0.5, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=50, trim_low=0.0, trim_high=0.0, Nsigma=1.0) 
    # Utils.plot_metrics_vs_feature(df=df_PbPb[(df_PbPb['MatchAttempts'] > 1000) & (df_PbPb['MatchAttempts']<2000)],feature=entry, threshold = 0.5, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=50, trim_low=0.0, trim_high=0.0, Nsigma=1.0)
    # Utils.plot_metrics_vs_feature(df=df_PbPb[df_PbPb['MatchAttempts']>3000],feature=entry, threshold = 0.5, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=50, trim_low=0.0, trim_high=0.0, Nsigma=1.0) 

#TODO: retrain smaller models on a specific snippet of featurespace to determine if we can squeeze out more performance by focussing on the especially challenging regions
# Current intended region: df_PbPb[df_PbPb['MatchAttempts']>3000]

# TODO - segment into low & high mult

# AUCPR_Evaluation of models

## On OO data

### OO model

In [ ]:
Evaluation.pr(y_pred=df_OO['score'],y_true=df_OO['IsSignal'], titleappendix="OO data, OO model")

### PbPb model

In [ ]:
Evaluation.pr(y_pred=df_OO['score_pbpb'],y_true=df_OO['IsSignal'], titleappendix="OO data, PbPb model")

## On PbPb data

### OO Model

In [ ]:
Evaluation.pr(y_pred=df_PbPb['score_oo'],y_true=df_PbPb['IsSignal'], titleappendix="PbPb data, OO model")

## PbPb Model

In [ ]:
Evaluation.pr(y_pred=df_PbPb['score'],y_true=df_PbPb['IsSignal'], titleappendix="PbPb data, PbPb model")
Evaluation.pr(y_pred=df_PbPb['score_xgb'],y_true=df_PbPb['IsSignal'], titleappendix="PbPb data, XGB model")

# Quick plots of dataset features

In [ ]:
df_OO['PMCH'].describe()

In [ ]:
feats2compare = ['PMCH','MatchAttempts', 'Rabs', 'RMFT','etaMFT']

In [ ]:
match_groups = Utils.build_match_groups(df_OO)
Utils.draw_all_features(feats2compare , match_groups, per=0.01, density=True, log=True)

In [ ]:
match_groups = Utils.build_match_groups(df_PbPb)
Utils.draw_all_features(feats2compare , match_groups, per=0.01, density=True, log=True)

# Appended own metrics

In [ ]:
Utils.pairing_purity_curve(df_OO, metric="score", n_thresholds=20)
Utils.pairing_purity_curve(df_OO, metric="score_pbpb", n_thresholds=20)


In [ ]:
importlib.reload(Utils)
Utils.pairing_purity_curve(df_PbPb, metric="score", n_thresholds=5)
Utils.pairing_purity_curve(df_PbPb, metric="score_oo", n_thresholds=5)
Utils.pairing_purity_curve(df_PbPb, metric="score_xgb", n_thresholds=5)


In [ ]:
importlib.reload(Utils)
Utils.pairing_purity_curve(df_PbPb, metric="score_nn", n_thresholds=10)

In [ ]:
Utils.pairing_purity_curve(df_PbPb, metric="score_xgb", n_thresholds=100)